In [3]:
import os
import time

import numpy as np
import scipy.io
import tensorflow as tf
from scipy.interpolate import griddata
from scipy.spatial import Delaunay
from scipy.optimize import minimize
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker

np.random.seed(1234)
tf.random.set_seed(1234)

# Plot style: readable in the thesis, consistent fonts
plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "savefig.dpi": 300,
})


# ══════════════════════════════════════════════════════════════════════════════
# PINN Model
# ══════════════════════════════════════════════════════════════════════════════

class PhysicsInformedNN:
    def __init__(self, x, y, t, u, v, layers, learning_rate=1e-3, dtype=tf.float32):
        self.dtype  = dtype
        self.layers = layers

        # ── Data ──────────────────────────────────────────────────────────────
        X = np.concatenate([x, y, t], axis=1).astype(np.float32)

        self.lb = tf.constant(X.min(0, keepdims=True), dtype=self.dtype)
        self.ub = tf.constant(X.max(0, keepdims=True), dtype=self.dtype)

        self.X = tf.convert_to_tensor(X, dtype=self.dtype)
        self.x = tf.convert_to_tensor(X[:, 0:1], dtype=self.dtype)
        self.y = tf.convert_to_tensor(X[:, 1:2], dtype=self.dtype)
        self.t = tf.convert_to_tensor(X[:, 2:3], dtype=self.dtype)

        self.u = tf.convert_to_tensor(u.astype(np.float32), dtype=self.dtype)
        self.v = tf.convert_to_tensor(v.astype(np.float32), dtype=self.dtype)

        # ── Network ───────────────────────────────────────────────────────────
        self.net = self._build_network(layers)
        _ = self.net(tf.zeros((1, layers[0]), dtype=self.dtype))  # build weights

        # ── Trainable PDE parameters (start at 0 — no hints) ─────────────────
        self.lambda_1 = tf.Variable(0.0, dtype=self.dtype, trainable=True, name="lambda_1")
        self.lambda_2 = tf.Variable(0.0, dtype=self.dtype, trainable=True, name="lambda_2")

        # ── Optimizer ─────────────────────────────────────────────────────────
        self.optimizer_adam = tf.keras.optimizers.Adam(learning_rate=learning_rate)

        # ── Training history (for the convergence figure) ─────────────────────
        self.history = {"it": [], "loss": [], "l1": [], "l2": []}
        self.n_adam = 0     # number of Adam iterations (position of the switch line)

    # ── Network architecture ──────────────────────────────────────────────────
    def _build_network(self, layers):
        model = tf.keras.Sequential(name="pinn")
        for width in layers[1:-1]:
            model.add(
                tf.keras.layers.Dense(
                    width,
                    activation=tf.nn.tanh,
                    kernel_initializer=tf.keras.initializers.GlorotNormal(),
                    bias_initializer="zeros",
                    dtype=self.dtype,
                )
            )
        model.add(
            tf.keras.layers.Dense(
                layers[-1],
                activation=None,
                kernel_initializer=tf.keras.initializers.GlorotNormal(),
                bias_initializer="zeros",
                dtype=self.dtype,
            )
        )
        return model

    @property
    def train_vars(self):
        return self.net.trainable_variables + [self.lambda_1, self.lambda_2]

    def neural_net(self, X):
        H = 2.0 * (X - self.lb) / (self.ub - self.lb) - 1.0
        return self.net(H)

    @staticmethod
    def _safe_grad(tape, y, x):
        g = tape.gradient(y, x)
        return tf.zeros_like(x) if g is None else g

    def _record(self, it, loss_value):
        self.history["it"].append(it)
        self.history["loss"].append(float(loss_value))
        self.history["l1"].append(float(self.lambda_1))
        self.history["l2"].append(float(self.lambda_2))

    # ── Physics residuals ─────────────────────────────────────────────────────
    def net_NS(self, x, y, t):
        x = tf.convert_to_tensor(x, dtype=self.dtype)
        y = tf.convert_to_tensor(y, dtype=self.dtype)
        t = tf.convert_to_tensor(t, dtype=self.dtype)

        with tf.GradientTape(persistent=True) as tape3:
            tape3.watch([x, y, t])
            with tf.GradientTape(persistent=True) as tape2:
                tape2.watch([x, y, t])
                with tf.GradientTape(persistent=True) as tape1:
                    tape1.watch([x, y, t])
                    X      = tf.concat([x, y, t], axis=1)
                    out    = self.neural_net(X)
                    psi    = out[:, 0:1]
                    p      = out[:, 1:2]

                u   = self._safe_grad(tape1, psi, y)
                v   = -self._safe_grad(tape1, psi, x)
                p_x = self._safe_grad(tape1, p, x)
                p_y = self._safe_grad(tape1, p, y)

            u_t = self._safe_grad(tape2, u, t)
            u_x = self._safe_grad(tape2, u, x)
            u_y = self._safe_grad(tape2, u, y)
            v_t = self._safe_grad(tape2, v, t)
            v_x = self._safe_grad(tape2, v, x)
            v_y = self._safe_grad(tape2, v, y)

        u_xx = self._safe_grad(tape3, u_x, x)
        u_yy = self._safe_grad(tape3, u_y, y)
        v_xx = self._safe_grad(tape3, v_x, x)
        v_yy = self._safe_grad(tape3, v_y, y)

        del tape1, tape2, tape3

        f_u = (u_t
               + self.lambda_1 * (u * u_x + v * u_y)
               + p_x
               - self.lambda_2 * (u_xx + u_yy))

        f_v = (v_t
               + self.lambda_1 * (u * v_x + v * v_y)
               + p_y
               - self.lambda_2 * (v_xx + v_yy))

        return u, v, p, f_u, f_v

    # ── Loss ──────────────────────────────────────────────────────────────────
    def loss(self, x, y, t, u, v):
        u_pred, v_pred, _, f_u_pred, f_v_pred = self.net_NS(x, y, t)
        return (tf.reduce_sum(tf.square(u - u_pred))
                + tf.reduce_sum(tf.square(v - v_pred))
                + tf.reduce_sum(tf.square(f_u_pred))
                + tf.reduce_sum(tf.square(f_v_pred)))

    # ── Adam ──────────────────────────────────────────────────────────────────
    @tf.function
    def train_step_adam(self):
        with tf.GradientTape() as tape:
            loss_value = self.loss(self.x, self.y, self.t, self.u, self.v)
        grads = tape.gradient(loss_value, self.train_vars)
        grads = [tf.zeros_like(v) if g is None else g
                 for g, v in zip(grads, self.train_vars)]
        self.optimizer_adam.apply_gradients(zip(grads, self.train_vars))
        return loss_value

    def train_adam(self, nIter):
        start_time = time.time()
        for it in range(nIter):
            loss_value = self.train_step_adam()
            if it % 10 == 0:
                self._record(it, loss_value)
                elapsed = time.time() - start_time
                print("It: %d, Loss: %.3e, l1: %.6f, l2: %.6f, Time: %.2fs"
                      % (it, float(loss_value),
                         float(self.lambda_1), float(self.lambda_2), elapsed))
                start_time = time.time()
        self.n_adam = nIter

    # ── L-BFGS-B ──────────────────────────────────────────────────────────────
    def _pack_weights(self):
        return np.concatenate(
            [tf.reshape(v, [-1]).numpy() for v in self.train_vars]
        ).astype(np.float64)

    def _unpack_weights(self, w):
        idx = 0
        for var in self.train_vars:
            size = int(np.prod(var.shape))
            var.assign(tf.constant(
                w[idx:idx + size].reshape(var.shape), dtype=self.dtype))
            idx += size

    def _loss_and_grad_lbfgs(self, w):
        self._unpack_weights(w)
        with tf.GradientTape() as tape:
            loss_value = self.loss(self.x, self.y, self.t, self.u, self.v)
        grads = tape.gradient(loss_value, self.train_vars)
        grads = [tf.zeros_like(v) if g is None else g
                 for g, v in zip(grads, self.train_vars)]
        grad_flat = np.concatenate(
            [tf.reshape(g, [-1]).numpy() for g in grads]).astype(np.float64)
        return float(loss_value), grad_flat

    def _callback_lbfgs(self, w):
        self.lbfgs_iter += 1
        self._unpack_weights(w)
        loss_value = float(self.loss(self.x, self.y, self.t, self.u, self.v))
        self._record(self.n_adam + self.lbfgs_iter, loss_value)
        print("L-BFGS-B Iter: %d/%d, Loss: %.3e, l1: %.6f, l2: %.6f"
              % (self.lbfgs_iter, self.lbfgs_maxiter,
                 loss_value, float(self.lambda_1), float(self.lambda_2)))

    def train_lbfgs(self, maxiter=10000, maxfun=15000,
                    maxcor=50, maxls=50, ftol=1.0 * np.finfo(float).eps):
        self.lbfgs_iter    = 0
        self.lbfgs_maxiter = maxiter
        result = minimize(
            fun=self._loss_and_grad_lbfgs,
            x0=self._pack_weights(),
            jac=True,
            method="L-BFGS-B",
            callback=self._callback_lbfgs,
            options={"maxiter": maxiter, "maxfun": maxfun,
                     "maxcor": maxcor,   "maxls":  maxls, "ftol": ftol},
        )
        self._unpack_weights(result.x)
        print("\nL-BFGS-B finished")
        print("  success :", result.success)
        print("  message :", result.message)
        print("  nit     :", result.nit)
        return result

    def train(self, adam_iters=10000, use_lbfgs=True, lbfgs_options=None):
        self.train_adam(adam_iters)
        if use_lbfgs:
            return self.train_lbfgs(**(lbfgs_options or {}))

    def predict(self, x_star, y_star, t_star, batch_size=8192):
        """Evaluate u, v, p in batches (the plotting grid has ~90 000 points)."""
        x_star = np.asarray(x_star, dtype=np.float32).reshape(-1, 1)
        y_star = np.asarray(y_star, dtype=np.float32).reshape(-1, 1)
        t_star = np.asarray(t_star, dtype=np.float32).reshape(-1, 1)
        u_out, v_out, p_out = [], [], []
        for i in range(0, x_star.shape[0], batch_size):
            s = slice(i, i + batch_size)
            u, v, p, _, _ = self.net_NS(x_star[s], y_star[s], t_star[s])
            u_out.append(u.numpy())
            v_out.append(v.numpy())
            p_out.append(p.numpy())
        return (np.concatenate(u_out), np.concatenate(v_out),
                np.concatenate(p_out))


# ══════════════════════════════════════════════════════════════════════════════
# Plotting helpers
# ══════════════════════════════════════════════════════════════════════════════

def inner_rectangle(X_star, nn=400):
    """Largest axis-aligned rectangle (approximately) that lies completely
    inside the region covered by the data points.

    The measurement points do not fill their bounding box: near the corners
    there are areas without data. Interpolating there gives NaN (white gaps),
    and filling them with nearest-neighbour values creates fake, blocky
    values. Instead, the plot is restricted to the part of the domain where
    data actually exists. The rectangle is shrunk from its worst side until
    every grid point is inside the convex hull of the data."""
    lb, ub = X_star.min(0), X_star.max(0)
    xi = np.linspace(lb[0], ub[0], nn)
    yi = np.linspace(lb[1], ub[1], nn)
    Xi, Yi = np.meshgrid(xi, yi)
    hull = Delaunay(X_star)
    inside = (hull.find_simplex(np.c_[Xi.ravel(), Yi.ravel()]) >= 0).reshape(Xi.shape)

    r0, r1, c0, c1 = 0, nn - 1, 0, nn - 1
    while True:
        sub = inside[r0:r1 + 1, c0:c1 + 1]
        if sub.all():
            break
        # fraction of points outside the data region on each border
        bad = {
            "bottom": np.mean(~sub[0, :]),
            "top":    np.mean(~sub[-1, :]),
            "left":   np.mean(~sub[:, 0]),
            "right":  np.mean(~sub[:, -1]),
        }
        side = max(bad, key=bad.get)
        if side == "bottom":
            r0 += 1
        elif side == "top":
            r1 -= 1
        elif side == "left":
            c0 += 1
        else:
            c1 -= 1
    return xi[c0], xi[c1], yi[r0], yi[r1]


def to_grid(X_star, values, Xi, Yi):
    """Cubic interpolation of scattered data onto the regular grid.
    The grid lies inside the data region, so NaN can only appear at
    isolated boundary points due to round-off; those few are filled
    with the nearest data value."""
    cubic = griddata(X_star, values, (Xi, Yi), method='cubic')
    if np.isnan(cubic).any():
        nearest = griddata(X_star, values, (Xi, Yi), method='nearest')
        cubic = np.where(np.isnan(cubic), nearest, cubic)
    return cubic


def plot_fields(extent, rows, title, out_base, show_error=True):
    """Predicted vs exact fields (and optionally |predicted - exact|).

    rows: list of (name, predicted_grid, exact_grid, cmap, rel_L2_error)
    The fields are drawn with imshow + bilinear interpolation, which gives a
    smooth colour map without contour bands or jagged edges."""
    x0, x1, y0, y1 = extent
    ncol = 3 if show_error else 2
    panel_w = 4.6
    panel_h = panel_w * (y1 - y0) / (x1 - x0)          # keep true aspect ratio
    fig, axes = plt.subplots(len(rows), ncol,
                             figsize=(ncol * panel_w + 1.5 * ncol,
                                      len(rows) * (panel_h + 0.9) + 0.6),
                             constrained_layout=True, squeeze=False)
    fig.suptitle(title)

    img_kw = dict(extent=extent, origin="lower", aspect="equal",
                  interpolation="bilinear")

    for r, (name, pred, exact, cmap, rel_err) in enumerate(rows):
        vmin = min(pred.min(), exact.min())
        vmax = max(pred.max(), exact.max())

        for c, (field, label) in enumerate([(pred, "predicted"),
                                            (exact, "exact")]):
            ax = axes[r, c]
            im = ax.imshow(field, cmap=cmap, vmin=vmin, vmax=vmax, **img_kw)
            ax.set_title(f"${name}$ {label}")
        cbar = fig.colorbar(im, ax=axes[r, :2], shrink=0.9, pad=0.02)
        cbar.locator = matplotlib.ticker.MaxNLocator(nbins=6)
        cbar.update_ticks()

        if show_error:
            ax = axes[r, 2]
            err = np.abs(pred - exact)
            im_e = ax.imshow(err, cmap="inferno", vmin=0.0, vmax=err.max(), **img_kw)
            ax.set_title(rf"$|{name}_{{pred}} - {name}_{{exact}}|$"
                         f"  (rel. $L_2$ = {100 * rel_err:.2f}%)", fontsize=10)
            cbar_e = fig.colorbar(im_e, ax=ax, shrink=0.9, pad=0.02)
            cbar_e.formatter = matplotlib.ticker.ScalarFormatter(useMathText=True)
            cbar_e.formatter.set_powerlimits((-2, 2))
            cbar_e.locator = matplotlib.ticker.MaxNLocator(nbins=5)
            cbar_e.update_ticks()

        for ax in axes[r, :]:
            ax.set_xlabel("$x$")
            ax.set_ylabel("$y$")
            ax.set_xlim(x0, x1)
            ax.set_ylim(y0, y1)

    fig.savefig(out_base + ".png", bbox_inches="tight", pad_inches=0.1)
    fig.savefig(out_base + ".pdf", bbox_inches="tight", pad_inches=0.1, dpi=300)
    plt.close(fig)
    print(f"  Figure saved: {out_base}.png / .pdf")


def plot_history(model, l2_true, title, out_base):
    """Loss and identified lambda_1, lambda_2 during training."""
    it   = np.array(model.history["it"])
    loss = np.array(model.history["loss"])
    l1   = np.array(model.history["l1"])
    l2   = np.array(model.history["l2"])

    fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
    fig.suptitle(title)

    axes[0].semilogy(it, loss, lw=1.2)
    axes[0].set_ylabel("loss")
    axes[1].plot(it, l1, lw=1.2, label=r"identified $\lambda_1$")
    axes[1].axhline(1.0, color="k", ls="--", lw=1, label="true value")
    axes[1].set_ylabel(r"$\lambda_1$")
    axes[2].plot(it, l2, lw=1.2, label=r"identified $\lambda_2$")
    axes[2].axhline(l2_true, color="k", ls="--", lw=1, label="true value")
    axes[2].set_ylabel(r"$\lambda_2$")

    # keep the true-value line inside the plot with some margin
    for ax, vals, true in [(axes[1], l1, 1.0), (axes[2], l2, l2_true)]:
        lo, hi = min(vals.min(), true), max(vals.max(), true)
        pad = 0.08 * (hi - lo if hi > lo else abs(true))
        ax.set_ylim(lo - pad, hi + pad)

    for ax in axes:
        ax.set_xlabel("iteration")
        if model.n_adam < it.max():
            ax.axvline(model.n_adam, color="gray", ls=":", lw=1)
        ax.grid(alpha=0.3)
    axes[1].legend(loc="lower right")
    axes[2].legend(loc="lower right")
    # mark where L-BFGS-B starts
    axes[0].text(model.n_adam, axes[0].get_ylim()[1], " L-BFGS-B",
                 va="top", ha="left", fontsize=9, color="gray")

    fig.savefig(out_base + ".png", bbox_inches="tight", pad_inches=0.1)
    fig.savefig(out_base + ".pdf", bbox_inches="tight", pad_inches=0.1)
    plt.close(fig)
    print(f"  Figure saved: {out_base}.png / .pdf")


def make_field_figures(model, x_star, y_star, t_snap,
                       u_ref, v_ref, p_ref, p_shift, errors, title, out_dir, tag):
    """Builds the plotting grid inside the data region, evaluates the PINN
    directly on that grid (no interpolation of the prediction), interpolates
    the reference data onto the same grid and draws the figures."""
    X_star = np.hstack([x_star[:, None], y_star[:, None]])   # N x 2
    x0, x1, y0, y1 = inner_rectangle(X_star)
    extent = (x0, x1, y0, y1)

    nn = 300
    xi = np.linspace(x0, x1, nn)
    yi = np.linspace(y0, y1, nn)
    Xi, Yi = np.meshgrid(xi, yi)

    # prediction: the network is a smooth function, evaluate it on the grid
    t_grid = np.full((Xi.size, 1), t_snap, dtype=np.float32)
    u_g, v_g, p_g = model.predict(Xi.reshape(-1, 1), Yi.reshape(-1, 1), t_grid)
    u_g = u_g.reshape(Xi.shape)
    v_g = v_g.reshape(Xi.shape)
    p_g = p_g.reshape(Xi.shape) + p_shift      # same pressure constant as in the errors

    # reference: scattered data -> grid (only inside the data region)
    u_e = to_grid(X_star, u_ref.flatten(), Xi, Yi)
    v_e = to_grid(X_star, v_ref.flatten(), Xi, Yi)
    p_e = to_grid(X_star, p_ref.flatten(), Xi, Yi)

    error_u, error_v, error_p = errors
    rows = [
        ("u", u_g, u_e, "RdBu_r", error_u),
        ("v", v_g, v_e, "RdBu_r", error_v),
        ("p", p_g, p_e, "viridis", error_p),
    ]
    # predicted | exact | error  (as in the original figure)
    plot_fields(extent, rows, title, os.path.join(out_dir, f"fields_{tag}"),
                show_error=True)
    # predicted | exact only
    plot_fields(extent, rows, title, os.path.join(out_dir, f"fields_compare_{tag}"),
                show_error=False)


# ══════════════════════════════════════════════════════════════════════════════
# Main
# ══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    # ── Config ────────────────────────────────────────────────────────────────
    MAT_FILE     = "cylinder_nektar_new_mid.mat"
    RE           = 100
    LAMBDA2_TRUE = 1.0 / RE
    N_train      = 5000
    layers       = [3, 20, 20, 20, 20, 20, 20, 20, 20, 2]
    ADAM_ITERS        = 10000
    LBFGS_OPTIONS     = {"maxiter": 10000, "maxfun": 50000}
    NOISY_ADAM_ITERS  = 60000
    NOISE             = 0.01

    # output names follow the data set, e.g. "cylinder_nektar_new_mid"
    tag     = os.path.splitext(os.path.basename(MAT_FILE))[0]
    out_dir = f"results_{tag}"
    os.makedirs(out_dir, exist_ok=True)

    print("=" * 60)
    print(f"  Dataset  : {MAT_FILE}")
    print(f"  Re       : {RE}   (true lambda_2 = {LAMBDA2_TRUE})")
    print(f"  N_train  : {N_train}")
    print(f"  Layers   : {layers}")
    print("=" * 60)

    # ── Load data ─────────────────────────────────────────────────────────────
    data = scipy.io.loadmat(MAT_FILE)

    # Keys of cylinder_nektar_new_mid.mat
    #   x_star, y_star : N x 1   spatial coordinates of the measurement points
    #   t_star         : T x 1   time instants
    #   U_star, V_star : N x T   velocity components u, v
    #   P_star         : N x T   pressure p
    x_star = data['x_star'].flatten()     # N,
    y_star = data['y_star'].flatten()     # N,
    t_star = data['t_star'].flatten()     # T,
    U_star = data['U_star']               # N x T
    V_star = data['V_star']               # N x T
    P_star = data['P_star']               # N x T

    N = x_star.shape[0]
    T = t_star.shape[0]

    print(f"\n  N points    : {N}")
    print(f"  T snapshots : {T}")
    print(f"  x range     : [{x_star.min():.3f}, {x_star.max():.3f}]")
    print(f"  y range     : [{y_star.min():.3f}, {y_star.max():.3f}]")
    print(f"  t range     : [{t_star.min():.3f}, {t_star.max():.3f}]\n")

    # ── Rearrange into flat (NT x 1) arrays ──────────────────────────────────
    XX = np.tile(x_star[:, None], (1, T))   # N x T
    YY = np.tile(y_star[:, None], (1, T))   # N x T
    TT = np.tile(t_star[None, :], (N, 1))   # N x T

    x = XX.flatten()[:, None]   # NT x 1
    y = YY.flatten()[:, None]   # NT x 1
    t = TT.flatten()[:, None]   # NT x 1
    u = U_star.flatten()[:, None]   # NT x 1
    v = V_star.flatten()[:, None]   # NT x 1
    p = P_star.flatten()[:, None]   # NT x 1

    # ── Random training subset ────────────────────────────────────────────────
    idx     = np.random.choice(N * T, N_train, replace=False)
    x_train = x[idx, :]
    y_train = y[idx, :]
    t_train = t[idx, :]
    u_train = u[idx, :]
    v_train = v[idx, :]

    ######################################################################
    ########################### Clean Data ###############################
    ######################################################################
    model = PhysicsInformedNN(x_train, y_train, t_train,
                              u_train, v_train, layers)
    model.train(adam_iters=ADAM_ITERS, use_lbfgs=True, lbfgs_options=LBFGS_OPTIONS)

    lambda_1_value = float(model.lambda_1)
    lambda_2_value = float(model.lambda_2)

    # ── Test at one snapshot (errors on the original data points) ─────────────
    snap = min(100, T - 1)

    x_test = x_star[:, None]                                  # N x 1
    y_test = y_star[:, None]                                  # N x 1
    t_test = np.full((N, 1), t_star[snap], dtype=np.float32)  # N x 1

    u_star_test = U_star[:, snap:snap + 1]   # N x 1
    v_star_test = V_star[:, snap:snap + 1]   # N x 1
    p_star_test = P_star[:, snap:snap + 1]   # N x 1

    u_pred, v_pred, p_pred = model.predict(x_test, y_test, t_test)

    # Pressure appears in the Navier-Stokes equations only through its
    # gradient, so the PINN can recover it only up to a constant. Shift the
    # predicted pressure so that its mean equals the mean of the reference.
    p_shift = p_star_test.mean() - p_pred.mean()
    p_pred  = p_pred + p_shift

    # ── Errors ────────────────────────────────────────────────────────────────
    error_u = np.linalg.norm(u_star_test - u_pred, 2) / np.linalg.norm(u_star_test, 2)
    error_v = np.linalg.norm(v_star_test - v_pred, 2) / np.linalg.norm(v_star_test, 2)
    error_p = np.linalg.norm(p_star_test - p_pred, 2) / np.linalg.norm(p_star_test, 2)

    error_lambda_1 = np.abs(lambda_1_value - 1.0) * 100
    error_lambda_2 = np.abs(lambda_2_value - LAMBDA2_TRUE) / LAMBDA2_TRUE * 100

    # ── Figures for the clean run ─────────────────────────────────────────────
    make_field_figures(
        model, x_star, y_star, t_star[snap],
        u_star_test, v_star_test, p_star_test, p_shift,
        errors=(error_u, error_v, error_p),
        title=(rf"$Re={RE}$, $t={t_star[snap]:.2f}$:  "
               rf"$\lambda_1={lambda_1_value:.4f}$, $\lambda_2={lambda_2_value:.5f}$"),
        out_dir=out_dir, tag=tag)
    plot_history(model, LAMBDA2_TRUE,
                 title=rf"Training history, $Re={RE}$",
                 out_base=os.path.join(out_dir, f"history_{tag}"))

    ######################################################################
    ########################### Noisy Data ###############################
    ######################################################################
    u_train_noisy = u_train + NOISE * np.std(u_train) * np.random.randn(*u_train.shape)
    v_train_noisy = v_train + NOISE * np.std(v_train) * np.random.randn(*v_train.shape)

    # Training with Adam only (no L-BFGS-B)
    model_noisy = PhysicsInformedNN(x_train, y_train, t_train,
                                    u_train_noisy, v_train_noisy, layers)
    model_noisy.train(adam_iters=NOISY_ADAM_ITERS, use_lbfgs=False)

    lambda_1_value_noisy = float(model_noisy.lambda_1)
    lambda_2_value_noisy = float(model_noisy.lambda_2)

    error_lambda_1_noisy = np.abs(lambda_1_value_noisy - 1.0) * 100
    error_lambda_2_noisy = np.abs(lambda_2_value_noisy - LAMBDA2_TRUE) / LAMBDA2_TRUE * 100

    ######################################################################
    ############################# Results ################################
    ######################################################################
    lines = [
        "=" * 60,
        f"  RESULTS  ({MAT_FILE}, Re = {RE})",
        "=" * 60,
        "  Clean data (Adam + L-BFGS-B):",
        f"    lambda_1 = {lambda_1_value:.6f}   (true = 1.0,  error = {error_lambda_1:.4f}%)",
        f"    lambda_2 = {lambda_2_value:.6f}   (true = {LAMBDA2_TRUE}, error = {error_lambda_2:.4f}%)",
        f"    rel. L2 error u : {error_u:.4e}",
        f"    rel. L2 error v : {error_v:.4e}",
        f"    rel. L2 error p : {error_p:.4e}   (after mean shift)",
        f"  Noisy data, {NOISE*100:.0f}% noise (Adam only, {NOISY_ADAM_ITERS} iterations):",
        f"    lambda_1 = {lambda_1_value_noisy:.6f}   (error = {error_lambda_1_noisy:.4f}%)",
        f"    lambda_2 = {lambda_2_value_noisy:.6f}   (error = {error_lambda_2_noisy:.4f}%)",
        "=" * 60,
    ]
    print("\n" + "\n".join(lines))
    with open(os.path.join(out_dir, f"results_{tag}.txt"), "w") as fh:
        fh.write("\n".join(lines) + "\n")
    print(f"  Results saved: {out_dir}/results_{tag}.txt")

  Dataset  : cylinder_nektar_new_mid.mat
  Re       : 100   (true lambda_2 = 0.01)
  N_train  : 5000
  Layers   : [3, 20, 20, 20, 20, 20, 20, 20, 20, 2]

  N points    : 1349
  T snapshots : 400
  x range     : [8.005, 12.988]
  y range     : [0.502, 2.494]
  t range     : [0.000, 39.900]



I0000 00:00:1792842388.984827   17335 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2015 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


It: 0, Loss: 4.255e+03, l1: -0.001000, l2: 0.001000, Time: 10.51s
It: 10, Loss: 8.700e+02, l1: 0.004738, l2: 0.009560, Time: 0.30s
It: 20, Loss: 6.939e+02, l1: 0.013390, l2: 0.013251, Time: 0.29s
It: 30, Loss: 5.448e+02, l1: 0.019164, l2: 0.013036, Time: 0.29s
It: 40, Loss: 4.833e+02, l1: 0.022973, l2: 0.007004, Time: 0.29s
It: 50, Loss: 4.407e+02, l1: 0.025811, l2: 0.002555, Time: 0.29s
It: 60, Loss: 4.107e+02, l1: 0.027885, l2: 0.000291, Time: 0.29s
It: 70, Loss: 3.852e+02, l1: 0.029294, l2: -0.000475, Time: 0.29s
It: 80, Loss: 3.647e+02, l1: 0.030439, l2: -0.000712, Time: 0.29s
It: 90, Loss: 3.469e+02, l1: 0.031558, l2: -0.000895, Time: 0.29s
It: 100, Loss: 3.317e+02, l1: 0.032708, l2: -0.001099, Time: 0.29s
It: 110, Loss: 3.192e+02, l1: 0.033870, l2: -0.001248, Time: 0.29s
It: 120, Loss: 3.092e+02, l1: 0.034962, l2: -0.001284, Time: 0.29s
It: 130, Loss: 3.015e+02, l1: 0.035901, l2: -0.001191, Time: 0.29s
It: 140, Loss: 2.960e+02, l1: 0.036641, l2: -0.000944, Time: 0.29s
It: 150, Lo